# Análises Prévias — Exploração dos Datasets Puros

Este notebook gera gráficos exploratórios para cada empresa (AGRO3, SLCE3, SOJA3) usando os datasets originais.

**Análises realizadas:**
1. **Força do Movimento** — Preço vs. Volume (volume confirma a tendência)
2. **Risco e Volatilidade** — Histograma de retornos diários
3. **Comparação de Séries Temporais** — Abertura, Fechamento, Alta e Baixa com destaque entre abertura/fechamento

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'legend.fontsize': 10,
})

In [ ]:
# ============================================================
# Carregar todos os datasets puros
# ============================================================
DATASETS_DIR = os.path.join(os.path.dirname(os.getcwd()), 'datasets')

EMPRESAS = {
    'AGRO3': {'nome': 'BrasilAgro (AGRO3)', 'cor': '#2196F3'},
    'SLCE3': {'nome': 'SLC Agrícola (SLCE3)', 'cor': '#4CAF50'},
    'SOJA3': {'nome': 'Boa Safra (SOJA3)', 'cor': '#FF9800'},
}

datasets = {}
for ticker in EMPRESAS:
    path = os.path.join(DATASETS_DIR, f'{ticker}.csv')
    df = pd.read_csv(path, parse_dates=['Date'])
    df = df.sort_values('Date').reset_index(drop=True)
    
    # Retorno diário percentual (Close-to-Close)
    df['Retorno_Diario'] = df['Close'].pct_change() * 100
    
    datasets[ticker] = df
    print(f'{ticker}: {len(df)} registros | {df["Date"].min().date()} até {df["Date"].max().date()}')

print(f'\nDatasets carregados: {list(datasets.keys())}')

---
## 1. Análise de Força do Movimento (Preço vs. Volume)

> O preço sozinho pode mentir, mas o volume confirma a tendência.
>
> Se as ações estão crescendo de forma constante, mas o volume de negociação está diminuindo dia após dia, significa que a alta está perdendo força (pouca gente sustentando o preço). Um crescimento forte deve ser acompanhado por um volume alto ou crescente.

In [ ]:
def plot_preco_volume(df, ticker, info):
    """
    Gráfico de preço de fechamento com barras de volume logo abaixo.
    As barras de volume são coloridas (verde = alta, vermelho = queda).
    """
    fig, (ax_price, ax_vol) = plt.subplots(
        2, 1, figsize=(16, 8), sharex=True,
        gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.05}
    )
    
    fig.suptitle(f'Força do Movimento — {info["nome"]}\nPreço de Fechamento vs. Volume',
                 fontsize=15, fontweight='bold', y=1.01)
    
    # --- Painel superior: Preço de fechamento ---
    ax_price.plot(df['Date'], df['Close'], color=info['cor'], linewidth=1.2, label='Fechamento')
    
    # Média móvel de 30 dias para tendência
    mm30 = df['Close'].rolling(30).mean()
    ax_price.plot(df['Date'], mm30, color='#E91E63', linewidth=1, linestyle='--',
                  alpha=0.7, label='MM 30 dias')
    
    ax_price.set_ylabel('Preço (R$)', fontweight='bold')
    ax_price.legend(loc='upper left')
    ax_price.tick_params(axis='x', labelbottom=False)
    
    # --- Painel inferior: Volume (barras coloridas) ---
    cores_vol = np.where(df['Close'] >= df['Open'], '#4CAF50', '#F44336')
    ax_vol.bar(df['Date'], df['Volume'], color=cores_vol, alpha=0.7, width=1.5)
    
    # Média móvel do volume
    mm_vol = df['Volume'].rolling(30).mean()
    ax_vol.plot(df['Date'], mm_vol, color='orange', linewidth=1.2, label='MM Volume 30d')
    
    ax_vol.set_ylabel('Volume', fontweight='bold')
    ax_vol.set_xlabel('Data', fontweight='bold')
    ax_vol.legend(loc='upper left', fontsize=9)
    ax_vol.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))
    
    # Formato de datas
    ax_vol.xaxis.set_major_locator(mdates.YearLocator())
    ax_vol.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax_vol.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
    plt.setp(ax_vol.xaxis.get_majorticklabels(), rotation=0)
    
    plt.tight_layout()
    fig.savefig(f'preco_volume_{ticker}.png', bbox_inches='tight', dpi=150)
    plt.show()


for ticker, info in EMPRESAS.items():
    plot_preco_volume(datasets[ticker], ticker, info)

---
## 2. Análise de Risco e Volatilidade (Histograma de Retornos Diários)

> Calculamos a variação percentual de um dia para o outro.
>
> Se a curva (distribuição) for muito larga e achatada, a ação é muito volátil — pode subir ou cair muito do nada.  
> Se for bem pontiaguda no zero, a ação é muito estável.

In [ ]:
def plot_histograma_retornos(df, ticker, info):
    """
    Histograma da variação percentual diária com curva normal sobreposta.
    Inclui estatísticas descritivas.
    """
    retornos = df['Retorno_Diario'].dropna()
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Histograma
    n_bins = min(80, int(np.sqrt(len(retornos)) * 2))
    n, bins, patches = ax.hist(
        retornos, bins=n_bins, density=True,
        color=info['cor'], alpha=0.6, edgecolor='white', linewidth=0.5,
        label='Retornos diários'
    )
    
    # Colorir barras: negativas em vermelho, positivas em verde
    for patch, left_edge in zip(patches, bins[:-1]):
        if left_edge < 0:
            patch.set_facecolor('#F44336')
            patch.set_alpha(0.55)
        else:
            patch.set_facecolor('#4CAF50')
            patch.set_alpha(0.55)
    
    # Curva normal teórica
    mu, sigma = retornos.mean(), retornos.std()
    x_range = np.linspace(retornos.min(), retornos.max(), 300)
    curva_normal = stats.norm.pdf(x_range, mu, sigma)
    ax.plot(x_range, curva_normal, color='#E91E63', linewidth=2.5,
            label=f'Normal (μ={mu:.2f}%, σ={sigma:.2f}%)')
    
    # Linha vertical na média
    ax.axvline(mu, color='#E91E63', linestyle='--', linewidth=1, alpha=0.8)
    ax.axvline(0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
    
    # Estatísticas no gráfico
    curtose = retornos.kurtosis()
    assimetria = retornos.skew()
    stats_text = (
        f'Média: {mu:.3f}%\n'
        f'Desvio Padrão: {sigma:.3f}%\n'
        f'Curtose: {curtose:.2f}\n'
        f'Assimetria: {assimetria:.2f}\n'
        f'Mín: {retornos.min():.2f}% | Máx: {retornos.max():.2f}%\n'
        f'Dias analisados: {len(retornos)}'
    )
    ax.text(0.98, 0.95, stats_text, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.9, edgecolor='gray'))
    
    # Interpretação da volatilidade
    if sigma > 4:
        vol_label = '⚠️ ALTA VOLATILIDADE'
    elif sigma > 2:
        vol_label = '⚡ VOLATILIDADE MODERADA'
    else:
        vol_label = '✅ BAIXA VOLATILIDADE'
    
    ax.set_title(f'Distribuição dos Retornos Diários — {info["nome"]}\n{vol_label}',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Retorno Diário (%)', fontweight='bold')
    ax.set_ylabel('Densidade', fontweight='bold')
    ax.legend(loc='upper left')
    
    plt.tight_layout()
    fig.savefig(f'histograma_retornos_{ticker}.png', bbox_inches='tight', dpi=150)
    plt.show()


for ticker, info in EMPRESAS.items():
    plot_histograma_retornos(datasets[ticker], ticker, info)

### Comparação de Volatilidade entre as 3 Empresas

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for ticker, info in EMPRESAS.items():
    retornos = datasets[ticker]['Retorno_Diario'].dropna()
    mu, sigma = retornos.mean(), retornos.std()
    x_range = np.linspace(-15, 15, 500)
    curva = stats.norm.pdf(x_range, mu, sigma)
    ax.plot(x_range, curva, linewidth=2.5, color=info['cor'],
            label=f'{ticker} (σ={sigma:.2f}%)')
    ax.fill_between(x_range, curva, alpha=0.1, color=info['cor'])

ax.axvline(0, color='black', linestyle='-', linewidth=0.8, alpha=0.4)
ax.set_title('Comparação de Volatilidade — Distribuição Normal dos Retornos Diários',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Retorno Diário (%)', fontweight='bold')
ax.set_ylabel('Densidade', fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
fig.savefig('comparacao_volatilidade.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 3. Comparação de Séries Temporais (Abertura, Fechamento, Alta, Baixa)

> Gráfico de série temporal com 4 linhas (Abertura, Fechamento, Alta, Baixa) em cores distintas.
> O espaço entre Abertura e Fechamento é destacado em cor transparente.

In [ ]:
def plot_serie_temporal_ohlc(df, ticker, info):
    """
    Série temporal com 4 linhas (Open, Close, High, Low).
    Área entre Open e Close preenchida com cor transparente.
    """
    fig, ax = plt.subplots(figsize=(18, 7))
    
    # Linhas principais
    ax.plot(df['Date'], df['High'], color='#4CAF50', linewidth=0.9,
            alpha=0.85, label='Alta (High)')
    ax.plot(df['Date'], df['Low'], color='#F44336', linewidth=0.9,
            alpha=0.85, label='Baixa (Low)')
    ax.plot(df['Date'], df['Open'], color='#2196F3', linewidth=1.1,
            alpha=0.9, label='Abertura (Open)')
    ax.plot(df['Date'], df['Close'], color='#FF9800', linewidth=1.1,
            alpha=0.9, label='Fechamento (Close)')
    
    # Preenchimento entre Abertura e Fechamento
    # Verde quando fechou acima da abertura, vermelho quando fechou abaixo
    ax.fill_between(
        df['Date'], df['Open'], df['Close'],
        where=(df['Close'] >= df['Open']),
        color='#4CAF50', alpha=0.12,
        interpolate=True, label='Close ≥ Open'
    )
    ax.fill_between(
        df['Date'], df['Open'], df['Close'],
        where=(df['Close'] < df['Open']),
        color='#F44336', alpha=0.12,
        interpolate=True, label='Close < Open'
    )
    
    ax.set_title(f'Série Temporal OHLC — {info["nome"]}\nAbertura, Fechamento, Alta e Baixa',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Data', fontweight='bold')
    ax.set_ylabel('Preço (R$)', fontweight='bold')
    
    ax.legend(loc='upper left', ncol=3, fontsize=9,
              framealpha=0.9, edgecolor='gray')
    
    # Formato de datas
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
    
    plt.tight_layout()
    fig.savefig(f'serie_temporal_ohlc_{ticker}.png', bbox_inches='tight', dpi=150)
    plt.show()


for ticker, info in EMPRESAS.items():
    plot_serie_temporal_ohlc(datasets[ticker], ticker, info)

---
## 4. Resumo Estatístico Geral

In [ ]:
resumo = []
for ticker, info in EMPRESAS.items():
    df = datasets[ticker]
    retornos = df['Retorno_Diario'].dropna()
    resumo.append({
        'Empresa': info['nome'],
        'Ticker': ticker,
        'Período': f"{df['Date'].min().strftime('%d/%m/%Y')} — {df['Date'].max().strftime('%d/%m/%Y')}",
        'Dias': len(df),
        'Preço Mín (R$)': df['Low'].min(),
        'Preço Máx (R$)': df['High'].max(),
        'Preço Médio (R$)': df['Close'].mean(),
        'Volume Médio': df['Volume'].mean(),
        'Retorno Médio Diário (%)': retornos.mean(),
        'Volatilidade (σ %)': retornos.std(),
        'Curtose': retornos.kurtosis(),
        'Assimetria': retornos.skew(),
        'Pior Dia (%)': retornos.min(),
        'Melhor Dia (%)': retornos.max(),
    })

df_resumo = pd.DataFrame(resumo).set_index('Ticker')
display(df_resumo.T)